# CyVerACT — Workflow runner

This notebook provides a common entry point for the Text-to-Cypher workflow, including its memory-enabled configurations. The same pipeline is used for individual questions, custom datasets and the thesis experiments.

| Mode | Dataset / input |
|---|---|
| `single_query` | One question against a Neo4j database |
| `custom_dataset` | An Excel dataset, with a shared database or connections per row |
| `movies_experiment` | Movies memory and test splits |
| `multidb_experiment` | MultiDB memory and test splits |

Execution is controlled in **Run settings**. With `RUN_MODE = None`, no workflow queries are started. Dataset population and evaluation are separate actions; neither is enabled by default.


## 1. Environment

The notebook runs from `CyVerACT/extended_workflow/`, where `graph.py` and the dataset folders are located. Install the repository dependencies (Python 3.11) and set up a local `.env` file:

```dotenv
NEO4J_URI=neo4j+ssc://YOUR_HOST:7687
NEO4J_USERNAME=neo4j
NEO4J_PASSWORD=YOUR_PASSWORD
OPENROUTER_API_KEY=YOUR_OPENROUTER_KEY
```

The `.env` connection is used by the **memory database** and, in `single_db` mode, by the target graph. For `per_row`, target database credentials are read from the Excel rows. The corresponding target graph(s) must exist before running queries.

For a new memory database, initialize its FixRules and indexes with `python init_memory_graph.py` and `python init_vectors.py`. **These scripts do not clear existing episodic memories.** An independent reproduction requires an appropriately isolated memory state.


In [ ]:
from pathlib import Path
import os
import re
import time

import pandas as pd
import torch
from dotenv import load_dotenv
from tqdm.auto import tqdm
from langchain_openai import ChatOpenAI

if not Path("graph.py").is_file():
    raise RuntimeError(
        "Open this notebook with CyVerACT/extended_workflow as the working directory."
    )

from graph import app

load_dotenv(override=True)

## 2. Run settings

Only these settings are needed for most runs. `RUN_MODE` selects the input; `DATASET_ACTION` selects **population** or **evaluation** when running a dataset. For a single question, `DATASET_ACTION` is not used.

`RUN_ID` identifies the saved dataset results. Keeping the same ID resumes an interrupted run; a new ID starts an independent run. `MAX_QUERIES` limits the **total number of dataset rows**, not the number of additional questions per run. `None` uses the whole file.

After changing the mode, model, configuration or run ID, rerun the settings and internal-configuration cells (and the model cell if the model changed) before executing a workflow cell.


In [ ]:
# Run selection
RUN_MODE = None  # None | "single_query" | "custom_dataset" | "movies_experiment" | "multidb_experiment"
DATASET_ACTION = None  # None | "populate" | "evaluate" (dataset modes only)

# Model and memory configuration
MODEL_LABEL = "llama_3_1_8b"  # llama_3_1_8b | llama_3_2_3b | qwen_2_5_7b
CONFIGURATION = "MEMORY_ALL_IN"  # BASELINE | HISTORY_ONLY | TRAJECTORY_ONLY | TRAJECTORY_HISTORY | GEN_HELPER_HISTORY | GEN_HELPER_TRAJ | MEMORY_ALL_IN

# Dataset output and resume
RUN_ID = "run_01"
MAX_QUERIES = None  # 2 for a quick check; None for the entire Excel file


### Input settings

These settings apply only to the corresponding mode. The Movies/MultiDB input files and connection modes are selected automatically.

- `single_query`: `SINGLE_QUESTION` and target database name.
- `custom_dataset`: Excel path and `single_db` / `per_row` connection mode.
- `movies_experiment`: the `movies` target database.

In `single_db`, all rows use the target database specified below. In `per_row`, every Excel row must include `question` (or `Question`), `url`, `database_name`, `username`, and `password`.


In [ ]:
# Single question (RUN_MODE="single_query")
SINGLE_QUESTION = "Who directed the movie The Matrix?"
SINGLE_SAVE_TO_MEMORY = False

# Shared target database (single_query / custom single_db / Movies)
TARGET_DATABASE_NAME = "movies"

# Custom Excel (RUN_MODE="custom_dataset")
CUSTOM_DATASET_PATH = "custom_smoke_queries.xlsx"
CUSTOM_CONNECTION_MODE = "single_db"  # "single_db" or "per_row"
CUSTOM_MEMORY_DATASET_PATH = None  # Separate Excel for population; None = no population dataset


### Internal configuration

The mappings below define the seven evaluated memory configurations and the file names used for results. They are not part of the per-run settings.


In [ ]:
# Internal mappings: 7 tested runtime memory configurations
MODEL_IDS = {
    "llama_3_1_8b": "meta-llama/llama-3.1-8b-instruct",
    "llama_3_2_3b": "meta-llama/llama-3.2-3b-instruct",
    "qwen_2_5_7b": "qwen/qwen-2.5-7b-instruct",
}

MEMORY_CONFIGURATIONS = {
    "BASELINE": dict(use_generator_helper=False, use_history=False, use_trajectory=False, use_fix_rules=False),
    "HISTORY_ONLY": dict(use_generator_helper=False, use_history=True, use_trajectory=False, use_fix_rules=True),
    "TRAJECTORY_ONLY": dict(use_generator_helper=False, use_history=False, use_trajectory=True, use_fix_rules=True),
    "TRAJECTORY_HISTORY": dict(use_generator_helper=False, use_history=True, use_trajectory=True, use_fix_rules=True),
    "GEN_HELPER_HISTORY": dict(use_generator_helper=True, use_history=True, use_trajectory=False, use_fix_rules=True),
    "GEN_HELPER_TRAJ": dict(use_generator_helper=True, use_history=False, use_trajectory=True, use_fix_rules=True),
    "MEMORY_ALL_IN": dict(use_generator_helper=True, use_history=True, use_trajectory=True, use_fix_rules=True),
}

CONFIG_FILE_SUFFIXES = {
    "BASELINE": "baseline",
    "HISTORY_ONLY": "history_only",
    "TRAJECTORY_ONLY": "traj_only",
    "TRAJECTORY_HISTORY": "traj_history",
    "GEN_HELPER_HISTORY": "gen_helper_history",
    "GEN_HELPER_TRAJ": "gen_helper_traj",
    "MEMORY_ALL_IN": "mem_all_in",
}

VALID_MODES = {None, "single_query", "custom_dataset", "movies_experiment", "multidb_experiment"}
if RUN_MODE not in VALID_MODES:
    raise ValueError(f"Invalid RUN_MODE: {RUN_MODE!r}. Choose None or one of {sorted(m for m in VALID_MODES if m is not None)}")
if MODEL_LABEL not in MODEL_IDS:
    raise ValueError(f"Unknown MODEL_LABEL: {MODEL_LABEL}")
if CONFIGURATION not in MEMORY_CONFIGURATIONS:
    raise ValueError(f"Unknown CONFIGURATION: {CONFIGURATION}")
if CUSTOM_CONNECTION_MODE not in {"single_db", "per_row"}:
    raise ValueError("CUSTOM_CONNECTION_MODE must be 'single_db' or 'per_row'")
if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_-]*", RUN_ID):
    raise ValueError("RUN_ID must contain only letters, digits, underscores or hyphens")
if MAX_QUERIES is not None and (not isinstance(MAX_QUERIES, int) or isinstance(MAX_QUERIES, bool) or MAX_QUERIES < 1):
    raise ValueError("MAX_QUERIES must be a positive integer or None")
VALID_ACTIONS = {None, "populate", "evaluate"}
if DATASET_ACTION not in VALID_ACTIONS:
    raise ValueError("DATASET_ACTION must be None, 'populate', or 'evaluate'")
if RUN_MODE == "single_query" and DATASET_ACTION is not None:
    raise ValueError("Set DATASET_ACTION=None for single_query mode")

MODEL_ID = MODEL_IDS[MODEL_LABEL]
MEMORY_FLAGS = MEMORY_CONFIGURATIONS[CONFIGURATION].copy()
CONFIG_FILE_SUFFIX = CONFIG_FILE_SUFFIXES[CONFIGURATION]
RUN_NAME = f"{MODEL_LABEL}_{CONFIGURATION}"
OUTPUT_NAME = f"{MODEL_LABEL}_{CONFIG_FILE_SUFFIX}_{RUN_ID}"
OUTPUT_ROOT = os.path.join("Outputs_Evaluation", "Reproduction_Runs", RUN_MODE or "_inactive")

TARGET_DATABASE_URI = os.getenv("NEO4J_URI")
TARGET_DATABASE_USER = os.getenv("NEO4J_USERNAME")
TARGET_DATABASE_PASSWORD = os.getenv("NEO4J_PASSWORD")

if RUN_MODE == "movies_experiment":
    TARGET_DATABASE_NAME = "movies"
    EVAL_DATASET = "Movies_DB_Datasets/100_test_queries_thesis.xlsx"
    POPULATION_DATASET = "Movies_DB_Datasets/50_memory_queries_thesis.xlsx"
    CONNECTION_MODE = "single_db"
elif RUN_MODE == "multidb_experiment":
    EVAL_DATASET = "Multi_DBs_Datasets/500_test_queries_thesis.xlsx"
    POPULATION_DATASET = "Multi_DBs_Datasets/250_memory_queries_thesis.xlsx"
    CONNECTION_MODE = "per_row"
elif RUN_MODE == "custom_dataset":
    EVAL_DATASET = CUSTOM_DATASET_PATH
    POPULATION_DATASET = CUSTOM_MEMORY_DATASET_PATH
    CONNECTION_MODE = CUSTOM_CONNECTION_MODE
else:
    EVAL_DATASET = None
    POPULATION_DATASET = None
    CONNECTION_MODE = "single_db"

print("Run mode:", RUN_MODE or "inactive — no questions will run")
print("Model:", MODEL_ID)
print("Configuration:", CONFIGURATION, MEMORY_FLAGS)
if RUN_MODE is not None:
    print("Output root:", OUTPUT_ROOT)
    print("Dataset:", EVAL_DATASET or "not used")
    print("Dataset action:", DATASET_ACTION or "none")


## 3. Model

The LLM runs through OpenRouter with `temperature=0`, `max_tokens=512`, and provider fallback enabled.

In [ ]:
if not os.getenv("OPENROUTER_API_KEY"):
    raise ValueError("OPENROUTER_API_KEY is missing from the environment/.env")

model = ChatOpenAI(
    model=MODEL_ID,
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
    temperature=0.0,
    max_tokens=512,
    extra_body={"provider": {"allow_fallbacks": True}},
)
print(f"Model initialized: {MODEL_ID}")

## 4. Workflow and dataset helpers

The next cell prepares LangGraph inputs, database connections, and incremental dataset outputs. 

Each processed row is saved to Excel and a pickle checkpoint. Output folders are separate from the archived thesis results. For `per_row`, each row provides its own Neo4j connection (`url`, `database_name`, `username`, `password`). These columns are retained in the results Excel, consistent with the original experiment files.


In [ ]:
def make_workflow_config():
    return {
        "configurable": {
            "model": model,
            "model_name": RUN_NAME,
            "attempts_w_filtered": 5,
            "total_attempts": 10,
            **MEMORY_FLAGS,
        },
        "recursion_limit": 50,
    }


def make_inputs(question_text, row=None, *, save_to_memory=False, connection_mode=None):
    mode = connection_mode or CONNECTION_MODE

    if not isinstance(question_text, str) or not question_text.strip():
        raise ValueError("Question must be a nonempty string")

    if mode == "single_db":
        details = {
            "database_url": TARGET_DATABASE_URI,
            "database_name": TARGET_DATABASE_NAME,
            "database_user": TARGET_DATABASE_USER,
            "database_password": TARGET_DATABASE_PASSWORD,
        }
    elif mode == "per_row":
        if row is None:
            raise ValueError("per_row connection mode requires a dataset row")
        required = ["url", "database_name", "username", "password"]
        missing = [key for key in required if key not in row or pd.isna(row[key])]
        if missing:
            raise ValueError(f"Dataset row lacks Neo4j connection columns: {missing}")
        details = {
            "database_url": row["url"],
            "database_name": row["database_name"],
            "database_user": row["username"],
            "database_password": row["password"],
        }
    else:
        raise ValueError(f"Invalid connection mode: {mode}")

    if any(not isinstance(value, str) or not value.strip() for value in details.values()):
        raise ValueError("Target Neo4j database connection parameters are missing")

    return {
        "question": question_text.strip(),
        **details,
        "save_to_memory": bool(save_to_memory),
    }


def evaluate_dataset(file_path, *, save_to_memory=False, max_queries=None):
    if not file_path or not Path(file_path).is_file():
        raise FileNotFoundError(f"Dataset does not exist: {file_path}")

    source = pd.read_excel(file_path).reset_index(drop=True)
    question_column = "question" if "question" in source.columns else "Question" if "Question" in source.columns else None
    if question_column is None:
        raise ValueError("Excel dataset needs a 'question' or 'Question' column")
    if CONNECTION_MODE == "per_row":
        missing = [c for c in ("url", "database_name", "username", "password") if c not in source.columns]
        if missing:
            raise ValueError(f"per_row dataset is missing these columns: {missing}")

    df = source.head(max_queries).copy() if max_queries is not None else source.copy()
    df = df.reset_index(drop=True)

    output_dir = Path(OUTPUT_ROOT) / ("memory_population" if save_to_memory else CONFIGURATION)
    checkpoint_dir = Path(OUTPUT_ROOT) / "temp" / ("population_" + OUTPUT_NAME if save_to_memory else OUTPUT_NAME)
    output_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)

    target_name = Path(file_path).stem.replace("_queries_thesis", "")
    phase_name = f"mem_{OUTPUT_NAME}" if save_to_memory else OUTPUT_NAME
    xlsx_path = output_dir / f"{target_name}_{phase_name}.xlsx"
    pkl_path = checkpoint_dir / f"{target_name}_{phase_name}.pkl"

    metrics = [
        "generated_cypher", "answer", "path", "output",
        "cypher_errors_history", "cypher_errors", "is_KG_Valid_correct",
        "is_KG_Valid_correct_with_full_schema", "total_tokens",
        "completion_tokens", "prompt_tokens", "eval_time_s",
    ]
    # Retain the dataset connection columns in the results (original workflow format).

    if xlsx_path.exists():
        results = pd.read_excel(xlsx_path).reset_index(drop=True).astype(object)
        saved_count = len(results)
        if saved_count > len(df):
            raise ValueError(
                f"Saved results contain {saved_count} rows, but MAX_QUERIES selects only {len(df)}. "
                "Increase MAX_QUERIES or use a different RUN_ID."
            )

        identity_columns = [question_column]
        if CONNECTION_MODE == "per_row":
            identity_columns.extend(["url", "database_name"])
            identity_columns.extend(c for c in ("username", "password") if c in results.columns)
        for column in identity_columns:
            if column not in results:
                raise ValueError(f"Existing results are missing '{column}'. Use a new RUN_ID.")
            previous = results[column].iloc[:saved_count].fillna("").astype(str).str.strip().tolist()
            current = df[column].iloc[:saved_count].fillna("").astype(str).str.strip().tolist()
            if previous != current:
                raise ValueError(
                    f"Input dataset changed in '{column}' compared with saved results. "
                    "Use a new RUN_ID for an independent run."
                )

        if saved_count < len(df):
            additional = df.iloc[saved_count:].astype(object)
            results = pd.concat([results, additional], ignore_index=True, sort=False).astype(object)
            print(f"Extending saved results: {saved_count} -> {len(results)} rows")
        else:
            print(f"Using saved results: {saved_count} rows")
    else:
        results = df.copy().astype(object)

    for column in metrics:
        if column not in results.columns:
            results[column] = None
    if CONNECTION_MODE == "per_row":
        for column in ("username", "password"):
            results[column] = df[column].to_numpy()

    completed = results["generated_cypher"].notna() & results["generated_cypher"].astype(str).str.strip().ne("")
    remaining = int((~completed).sum())
    print(f"Dataset: {len(source)} rows; selected: {len(df)}; already completed: {int(completed.sum())}; remaining: {remaining}")
    print(f"Results: {xlsx_path}")
    if remaining == 0:
        # An older checkpoint may have been saved without the connection columns.
        # Persist the backfilled columns even when no workflow calls are needed.
        if CONNECTION_MODE == "per_row":
            results.to_pickle(pkl_path)
            results.to_excel(xlsx_path, index=False)
        print("No new questions to run. Increase MAX_QUERIES (if the Excel has more rows) or choose a new RUN_ID.")
        return results

    workflow_config = make_workflow_config()
    executed = 0
    for i, row in tqdm(results.iterrows(), total=len(results), desc="Evaluating", unit="query"):
        if completed.iloc[i]:
            continue
        source_row = df.iloc[i]  # Per-row connection values come from the source Excel
        inputs = make_inputs(source_row[question_column], row=source_row, save_to_memory=save_to_memory)

        start = time.perf_counter()
        result = app.invoke(inputs, config=workflow_config)
        elapsed = time.perf_counter() - start

        path_list = result.get("path", [])
        is_valid_correct = result.get("is_KG_Valid_correct", False)
        is_full_schema_correct = ("schema_retriever" in path_list) and bool(is_valid_correct)

        results.at[i, "eval_time_s"] = round(elapsed, 4)
        results.at[i, "generated_cypher"] = result.get("cypher_statement", "")
        results.at[i, "output"] = str(result.get("database_records", ""))
        results.at[i, "path"] = str(path_list)
        results.at[i, "cypher_errors"] = str(result.get("cypher_errors", []))
        results.at[i, "cypher_errors_history"] = str(result.get("cypher_errors_history", []))
        results.at[i, "is_KG_Valid_correct"] = is_valid_correct
        results.at[i, "is_KG_Valid_correct_with_full_schema"] = is_full_schema_correct
        results.at[i, "answer"] = result.get("answer", "")
        results.at[i, "total_tokens"] = str(result.get("total_tokens", ""))
        results.at[i, "prompt_tokens"] = str(result.get("prompt_tokens", ""))
        results.at[i, "completion_tokens"] = str(result.get("completion_tokens", ""))
        executed += 1

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        results.to_pickle(pkl_path)
        results.to_excel(xlsx_path, index=False)
        time.sleep(1)  # Inter-query delay: excluded from eval_time_s.

    print(f"Newly executed: {executed}; total selected: {len(results)}. Results: {xlsx_path}")
    return results


## 5. Execute

The execution cells below share the settings from Section 2. Only the selected mode/action runs; the other sections are skipped.


### Single question

With `RUN_MODE="single_query"`, this cell executes `SINGLE_QUESTION` against `TARGET_DATABASE_NAME`. `SINGLE_SAVE_TO_MEMORY=False` leaves the episodic memory unchanged.


In [ ]:
if RUN_MODE != "single_query":
    print("Single-query section skipped (choose RUN_MODE='single_query' to enable it).")
else:
    inputs = make_inputs(SINGLE_QUESTION, save_to_memory=SINGLE_SAVE_TO_MEMORY)
    result = app.invoke(inputs, config=make_workflow_config())

    print("\nQuestion:", SINGLE_QUESTION)
    print("Generated Cypher:", result.get("cypher_statement", ""))
    print("Answer:", result.get("answer", ""))
    print("KG valid:", result.get("is_KG_Valid_correct", False))
    records = (len(result.get("database_records", []) or [])
               if "cypher_executor" in result.get("path", []) else 0)
    print("Records:", records)
    print("Workflow path:", result.get("path", []))

### Dataset population

For dataset modes, set `DATASET_ACTION="populate"` and run this cell. Population writes successful patterns and failed attempts to the long-term memory database. For `custom_dataset`, a separate `CUSTOM_MEMORY_DATASET_PATH` is required. `BASELINE` does not use long-term retrieval and skips population.

The memory database is **not reset automatically**. A population checkpoint is meaningful only while its corresponding memory graph is retained. Evaluation questions should not be used as population examples.


In [ ]:
if RUN_MODE not in {"custom_dataset", "movies_experiment", "multidb_experiment"} or DATASET_ACTION != "populate":
    print("Population not selected.")
elif CONFIGURATION == "BASELINE":
    print("BASELINE does not require memory population.")
elif not POPULATION_DATASET:
    raise ValueError("Population dataset not set. Specify CUSTOM_MEMORY_DATASET_PATH for custom runs.")
else:
    memory_population_results = evaluate_dataset(
        POPULATION_DATASET, save_to_memory=True, max_queries=MAX_QUERIES
    )


### Dataset evaluation

For dataset modes, set `DATASET_ACTION="evaluate"` and run this cell. Evaluation reads available memory but **does not save new episodes** (`save_to_memory=False`).

Results are saved after each question under `Outputs_Evaluation/Reproduction_Runs/`. A repeated `RUN_ID` resumes completed questions. Increasing `MAX_QUERIES` from `2` to `None` continues with the remaining rows **if the Excel contains more than two rows**; the first two are not rerun. Changing the dataset or the experimental setup requires a new `RUN_ID`.


In [ ]:
if RUN_MODE not in {"custom_dataset", "movies_experiment", "multidb_experiment"} or DATASET_ACTION != "evaluate":
    print("Evaluation not selected.")
else:
    dataset_results = evaluate_dataset(
        EVAL_DATASET, save_to_memory=False, max_queries=MAX_QUERIES
    )


## 6. Dataset reference

| Experiment | Population dataset | Evaluation dataset | Target connection |
|---|---|---|---|
| Movies | `Movies_DB_Datasets/50_memory_queries_thesis.xlsx` | `Movies_DB_Datasets/100_test_queries_thesis.xlsx` | `.env`, database `movies` |
| MultiDB | `Multi_DBs_Datasets/250_memory_queries_thesis.xlsx` | `Multi_DBs_Datasets/500_test_queries_thesis.xlsx` | `per_row` Excel columns |
| Custom | `CUSTOM_MEMORY_DATASET_PATH` (optional) | `CUSTOM_DATASET_PATH` | `single_db` or `per_row` |

**Excel format:** `question` (or `Question`) is required in every dataset. `per_row` additionally needs `url`, `database_name`, `username`, `password`. In `per_row` mode, the source connection columns are also kept in the output Excel, as in the original workflow results. Review these files before sharing them outside the demo environment.

**Reproduction sequence:** prepare the target graph and isolated memory database; populate using the memory split (`DATASET_ACTION="populate"`); evaluate using the test split (`DATASET_ACTION="evaluate"`). Keep model/configuration consistent across both phases. Use a new `RUN_ID` and an appropriate memory state for each independent experiment. Baseline needs no memory population.

`MAX_QUERIES` limits the selected prefix of each dataset. For a two-question Excel, `MAX_QUERIES=None` still runs only two questions. Stored results are not a guarantee of identical historical metrics: DB snapshots, provider/model behavior and the memory state also matter.

### Memory configurations

| Configuration | Generator Helper | Short-term History | Trajectory | FixRules |
|---|:---:|:---:|:---:|:---:|
| `BASELINE` | — | — | — | — |
| `HISTORY_ONLY` | — | ✓ | — | ✓ |
| `TRAJECTORY_ONLY` | — | — | ✓ | ✓ |
| `TRAJECTORY_HISTORY` | — | ✓ | ✓ | ✓ |
| `GEN_HELPER_HISTORY` | ✓ | ✓ | — | ✓ |
| `GEN_HELPER_TRAJ` | ✓ | — | ✓ | ✓ |
| `MEMORY_ALL_IN` | ✓ | ✓ | ✓ | ✓ |
